<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c35-header.png" alt="Nextia Learning · Forecasting and Time Series: Predict What Comes Next" width="100%">

# Intermittent demand

**[Intermittent demand](https://learning.nextia-ai.com/courses/forecasting/m06/intermittent-demand/)** · Forecasting and Time Series: Predict What Comes Next · Module 6, lesson 2 of 4 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** find the slow sellers, forecast them with Croston, SBA and TSB, and see why the usual error measures choose the wrong forecast for them.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key. |
| **You should know** | MASE, WAPE and the stock cost, from [Measure the error that matters](https://learning.nextia-ai.com/courses/forecasting/m03/measure-the-error-that-matters/). The backtest, from [Test like the future](https://learning.nextia-ai.com/courses/forecasting/m03/test-like-the-future-backtesting/). |
| **Tested** | Python 3.12 with pandas 2.3.3, statsforecast 2.1.1, mlforecast 1.1.0, hierarchicalforecast 1.5.3, LightGBM 4.7.0 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m06/intermittent-demand/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C35/M06-L02-intermittent-demand/intermittent-demand-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 2.3.3, statsforecast 2.1.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==2.3.3 statsforecast==2.1.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import statsforecast; print("statsforecast", statsforecast.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield data of this course (about 2 MB) into a folder `larkfield-forecasting/data`, and the course's backtest helper `backtest.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 403758 rows of daily sales (200 products, 3 regions), from 2024-04-01 to 2026-03-29.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-forecasting` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield data is synthetic: we made it for this course. It has no real people, products or prices.

In [ ]:
# Setup: download the course data into larkfield-forecasting/data/ and check it.
import gzip, hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C35/"
FILES = {
    "calendar.csv": "a5d1d5a3b640a983de72a30fca0e627a4ef23e797b1d80619e9ffcaadf118844",
    "demand.csv.gz": "8b39303985cbdb9bb1f3c705cd927ed6aaeb6f9665bd56ad4c3f8cb2b8578c99",
    "orders.csv": "bd8059fd73fe294daa157f8a6b6ae3f58319fa644eaa7ff0d31122b02cf05e30",
    "price_changes.csv": "889b258be4e933b87065d81d0771d6cd9b630ee90223b0a6ab074129c42153cb",
    "products.csv": "198b0a3de6eea8128af68f3cbc3a5c6acd3a64ac854fff18d37660857be8cfbf",
    "promotions.csv": "ca147959faa4c36c757742de7121976ffe79d59b53049dbbc8337a7547541f4a",
    "roster_hourly.csv": "6c2e0e96881b36ded52ed19458515c71f025288ddaebd589b50c82a8ad7fa69b",
    "tickets_daily.csv": "36f7067db060fdd2bd084c408c0622370a012773849f243020336f1fc678b31d",
    "tickets_hourly.csv.gz": "43f7fbb3e360a8e4a0cb4e101ea6a6242a76f0c37204d76ee9413ea3adbedd92",
    "weather.csv": "43a199a8b235940e291195bfd026901b62a10189b29f3a3ad9527b5382544276",
    "backtest.py": "764ecd38c6e96632be6f0e139e05eadd7b07d14a4ece73d881fc7559c2fa203a",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-forecasting").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected):
    """Use a copy next to the notebook if there is one, or download it."""
    if path.exists() and sha256(path) == expected:
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-forecasting and run again.")

for name, expected in FILES.items():
    if name == "backtest.py":
        fetch(name, LABS + name, PROJECT / name, expected)
    else:
        fetch(name, LABS + "data/" + name, PROJECT / "data" / name, expected)

os.chdir(PROJECT)  # work in the project folder, as in the lesson
with gzip.open("data/demand.csv.gz", "rt") as f:
    header = f.readline()
    rows = [line.split(",", 3)[:3] for line in f]
print(f"Ready: {len(rows)} rows of daily sales ({len({r[2] for r in rows})} products, {len({r[1] for r in rows})} regions), "
      f"from {min(r[0] for r in rows)} to {max(r[0] for r in rows)}.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course's backtest helper as `bt`, and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the backtest helper, and checks that do not show the answer.
import hashlib, json, logging, os, warnings
os.environ["NIXTLA_ID_AS_COL"] = "1"
import numpy as np
import pandas as pd
import backtest as bt
pd.set_option("display.precision", 3)
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)
warnings.filterwarnings("ignore")
logging.getLogger("lightning").setLevel(logging.ERROR)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

def load_sales(products=None):
    """Daily sales per product, the three regions added up, in the long format of the backtest:
    unique_id (the product), ds (the day), y (units sold), price, on_promotion, stock_out.
    The missing day 2025-10-19 gets the mean of the same weekday one week before and after."""
    d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
    if products is not None:
        d = d[d["product_id"].isin(products)]
    g = d.groupby(["product_id", "date"], as_index=False).agg(
        y=("sales", "sum"), price=("price", "mean"), on_promotion=("on_promotion", "max"), stock_out=("stock_out", "max"))
    out = []
    for uid, s in g.rename(columns={"product_id": "unique_id", "date": "ds"}).groupby("unique_id"):
        s = s.set_index("ds").reindex(pd.date_range(s["ds"].min(), s["ds"].max(), freq="D"))
        gap = s["y"].isna()
        before, after = s["y"].shift(7), s["y"].shift(-7)
        s.loc[gap, "y"] = ((before + after) / 2).fillna(before).fillna(after)[gap].round()
        s = s.ffill()
        s["unique_id"] = uid
        out.append(s.rename_axis("ds").reset_index())
    return pd.concat(out, ignore_index=True)[["unique_id", "ds", "y", "price", "on_promotion", "stock_out"]]

# The course backtest (the same in every module): 13 origins, 28 days apart, the last one 2026-03-15.
ORIGINS = bt.origins("2026-03-15", n=13, every="28D")
BACKTEST_PRODUCTS = ['P001', 'P002', 'P005', 'P006', 'P009', 'P013', 'P018', 'P024', 'P025', 'P029', 'P032', 'P039', 'P045', 'P057', 'P068', 'P075', 'P077', 'P078', 'P080', 'P081', 'P087', 'P088', 'P093', 'P095', 'P108', 'P110', 'P121', 'P130', 'P133', 'P136', 'P137', 'P141', 'P145', 'P147', 'P148', 'P156', 'P158', 'P159', 'P166', 'P169', 'P172', 'P174', 'P176', 'P183', 'P184', 'P186', 'P191', 'P192', 'P194', 'P199']
PRICES = pd.read_csv("data/products.csv").set_index("product_id")["base_price"].to_dict()

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. Croston by hand

A slow seller sold these units in 14 days: `0 0 2 0 0 0 1 0 0 3 0 0 0 0`. Croston's method splits the series into two smaller series:

- the **demand size**: the units on each day with a sale (2, 1, 3);
- the **interval**: the days from one sale to the next (3, 4, 3).

It smooths each one with simple exponential smoothing (weight 0.1 for the newest value), then divides: units per sale ÷ days per sale = units per day.

> **Predict.** The plain averages are 2 units and 3.33 days. Will Croston's forecast per day be above or below 2 ÷ 3.33 = 0.6? Then run the cell.

In [ ]:
def smooth(values, alpha=0.1):
    """Simple exponential smoothing: start at the first value, then move 10% toward each new value."""
    level = values[0]
    for v in values[1:]:
        level = alpha * v + (1 - alpha) * level
    return level

def croston(y):
    y = np.asarray(y, dtype=float)
    days = np.flatnonzero(y > 0)               # the days with a sale (0 = the first day)
    sizes = y[days]                            # units on each of those days
    intervals = np.diff(days + 1, prepend=0)   # days since the sale before (the first: since the start)
    size, interval = smooth(sizes), smooth(intervals)
    return size, interval, size / interval

tiny = [0, 0, 2, 0, 0, 0, 1, 0, 0, 3, 0, 0, 0, 0]
size, interval, per_day = croston(tiny)
print(f"size {size:.2f}, interval {interval:.2f}, forecast {per_day:.3f} a day")

from statsforecast import StatsForecast
from statsforecast.models import TSB, CrostonClassic, CrostonSBA
t = pd.DataFrame({"unique_id": "tiny", "ds": pd.date_range("2025-06-02", periods=14), "y": tiny})
f = StatsForecast(models=[CrostonClassic(), CrostonSBA(), TSB(alpha_d=0.1, alpha_p=0.1)], freq="D").forecast(df=t, h=1)
print(f[["CrostonClassic", "CrostonSBA", "TSB"]].round(3).to_string(index=False))

> **Check.** You should see `size 2.01, interval 3.09, forecast 0.650 a day`, and statsforecast's `CrostonClassic` gives the same (0.65). SBA is Croston × 0.95 (0.618). TSB gives 0.291: it also counts the four zero days at the end, and Croston does not.

## 2. Find the slow sellers

The bottom of the hierarchy is one product in one region. The next cell keeps the series on sale for the whole history with **70% or more zero days**, and fills the missing day as in Module 2.

In [ ]:
d = pd.read_csv("data/demand.csv.gz", parse_dates=["date"])
products = pd.read_csv("data/products.csv", parse_dates=["launch_date"])
pr = d.assign(unique_id=d["product_id"] + "/" + d["region"]).rename(columns={"date": "ds", "sales": "y"})
pr = pr[["unique_id", "ds", "y"]].set_index(["unique_id", "ds"]).sort_index()
# every day for every series: the missing day gets the mean of the same weekday a week before and after
full = pd.MultiIndex.from_product([pr.index.levels[0], pd.date_range(pr.index.levels[1].min(), "2026-03-29")],
                                  names=["unique_id", "ds"])
pr = pr.reindex(full).reset_index()
pr = pr[pr.groupby("unique_id")["y"].transform(lambda s: s.ffill().notna())]  # from the first day of each series
g = pr.groupby("unique_id")["y"]
pr["y"] = pr["y"].fillna(((g.shift(7) + g.shift(-7)) / 2).round())

on_sale_all_history = set(products.loc[products["launch_date"] == "2024-04-01", "product_id"])
zero_share = pr.groupby("unique_id")["y"].apply(lambda s: (s == 0).mean())
slow_ids = sorted(u for u, z in zero_share.items() if z >= 0.7 and u.split("/")[0] in on_sale_all_history)
slow = pr[pr["unique_id"].isin(slow_ids)].reset_index(drop=True)
print(len(slow_ids), "slow series | mean share of zero days:", round(zero_share[slow_ids].mean(), 3))
print("vacuum bags P170, days with a sale:", {r: int((pr[pr["unique_id"] == f"P170/{r}"]["y"] > 0).sum()) for r in ["north", "south", "coast"]})

> **Check.** You should see `140 slow series | mean share of zero days: 0.902` and `{'north': 12, 'south': 5, 'coast': 11}`: the vacuum bags sell on about one day a month.

## 3. Backtest the slow-seller methods

The course backtest: 13 cut-offs, 14 days each. Six methods: always 0, seasonal naive, Croston, SBA, TSB and ADIDA. ADIDA adds the days up into blocks as long as the average interval, forecasts the blocks, and spreads the result evenly over the days. The cost is the 14-day stock order of each series, as in Module 3.

> **Predict.** Which method gets the best (lowest) MASE? Which gets the lowest cost? Then run the cell (a few seconds).

In [ ]:
from statsforecast.models import ADIDA, SeasonalNaive, ZeroModel

def stats(make_model):
    """A statsforecast model as a forecaster for bt.backtest()."""
    def forecaster(train, h, future):
        model = make_model()
        f = StatsForecast(models=[model], freq="D", n_jobs=1).forecast(df=train[["unique_id", "ds", "y"]], h=h)
        return pd.DataFrame({"unique_id": f["unique_id"], "ds": f["ds"], "yhat": f[str(model)].clip(lower=0)})
    return forecaster

methods = {"always 0": ZeroModel, "seasonal naive": lambda: SeasonalNaive(season_length=7),
           "Croston": CrostonClassic, "SBA": CrostonSBA, "TSB": lambda: TSB(alpha_d=0.1, alpha_p=0.1),
           "ADIDA": ADIDA}
cost = bt.stock_cost({u: PRICES[u.split("/")[0]] for u in slow_ids})
rows, forecasts = [], {}
for name, make in methods.items():
    forecasts[name] = bt.backtest(slow, stats(make), h=14, origins=ORIGINS)
    rows.append(bt.score(forecasts[name], slow, name, season=7, cost=cost))
table = pd.DataFrame(rows)[["method", "MASE", "WAPE %", "bias %", "cost $"]]
print(table.to_string(index=False))

> **Check.** You should see six rows. `always 0` has the best MASE (0.599) but a WAPE of 100%, a bias of −100% and a cost of 4371 dollars. The seasonal naive forecast has the lowest cost (2885), then ADIDA (2911). Croston costs 3733 and SBA 3679.

The measure and the decision disagree. On a slow seller, 9 days in 10 have no sale. The forecast that misses the fewest units on a typical day is zero. But an order of zero loses every sale.

## 4. Why "always 0" wins on MASE

MASE divides the mean absolute error by a scale. On a day with no sale, a forecast of 0 has no error, and Croston's 0.15 has an error of 0.15. When most days are zero, those small errors add up.

In [ ]:
f = pd.concat(forecasts.values())
test_days = forecasts["always 0"]
print("forecast days:", len(test_days), "| days with no sale:", int((test_days["y"] == 0).sum()),
      f"({(test_days['y'] == 0).mean():.1%})", "| units sold:", int(test_days["y"].sum()))
for name in ["always 0", "Croston"]:
    fc = forecasts[name]
    print(f"{name:9} mean absolute error per day: {(fc['y'] - fc['yhat']).abs().mean():.4f}")

> **Check.** You should see `forecast days: 25480 | days with no sale: 22993 (90.2%) | units sold: 3120`, a mean absolute error of 0.1224 for always 0 and 0.2026 for Croston.

## 5. Your turn: the frying pan

> **Your turn.** The frying pan in the north (`"P120/north"`) is one of the slow series. Take its sales up to the last cut-off (`ORIGINS[-1]`, 15 March 2026), apply the `croston()` function of section 1, and put the **forecast per day**, rounded to 3 decimals, in `pan_per_day`. Then run the check cell.

In [ ]:
pan_per_day = ...  # Croston's forecast per day for P120/north

In [ ]:
expect('pan_per_day', pan_per_day, 0.146, tolerance=0.0005)

The frying pan sells about 1.07 units every 7.32 days. Croston's forecast is 0.146 a day, about 2 units in 14 days. Its last sale was 15 days before the cut-off, but Croston does not see that: it changes only on a day with a sale.

## 6. Failure case: the best score, the worst order

If you choose the method by MASE alone, you choose "always 0", and Kenji orders nothing for 140 slow series. In the backtest, that order cost 4371 dollars per order cycle in lost sales, against 2885 for the seasonal naive order. For slow sellers, choose by the cost of the decision, and look at the bias.

## Next

You can now forecast slow sellers and choose them by their cost. The [lesson page](https://learning.nextia-ai.com/courses/forecasting/m06/intermittent-demand/) has the diagrams, the full table with ETS and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [New products with no history](https://learning.nextia-ai.com/courses/forecasting/m06/new-products-with-no-history/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/forecasting/m06/intermittent-demand/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.